# Lab 5: Implementation of NP Hard and NP Complete Problems

## Objective of the Lab

- To understand the concepts of **P**, **NP**, **NP-Hard**, and **NP-Complete** complexity classes.
- To understand **Cook's Theorem (Cook-Levin Theorem)** and the role of reductions in proving NP-completeness.
- To implement classical NP-Hard / NP-Complete problems and observe their exponential-time behavior through brute-force / backtracking solutions.

## Related Theory

**P** is the class of decision problems that can be solved in polynomial time by a deterministic algorithm.

**NP** (Nondeterministic Polynomial time) is the class of decision problems for which a given solution (a *certificate*) can be **verified** in polynomial time, even if finding that solution may take much longer.

**NP-Hard** problems are at least as hard as the hardest problems in NP  every problem in NP can be reduced to an NP-Hard problem in polynomial time. NP-Hard problems need not themselves be in NP (they may not even be decision problems, e.g. optimization problems).

**NP-Complete** problems are the intersection of NP and NP-Hard: they are in NP (verifiable in polynomial time) **and** every other problem in NP can be reduced to them in polynomial time. If any NP-Complete problem could be solved in polynomial time, then P = NP.

**Cook's Theorem (Cook-Levin Theorem, 1971)** was the first and most fundamental NP-Completeness result. It proves that the **Boolean Satisfiability Problem (SAT)** — determining whether there exists an assignment of true/false values to variables that makes a given Boolean formula true  is NP-Complete. Every problem in NP can be reduced to SAT in polynomial time, which is why SAT is considered the "origin" of NP-Completeness; all other NP-Complete problems (Vertex Cover, Hamiltonian Cycle, Subset Sum, etc.) are proven NP-Complete by reducing SAT (or another already-proven NP-Complete problem) to them.

Some well-known NP-Hard / NP-Complete problems covered in this lab:

- **Hamiltonian Cycle**: Does a graph contain a cycle that visits every vertex exactly once and returns to the start? (NP-Complete graph problem.)
- **Minimum Makespan Scheduling**: Given jobs with processing times and a fixed number of identical machines, assign jobs to machines to minimize the maximum completion time (makespan). (NP-Hard scheduling problem, related to the Partition Problem.)
- **Register Allocation (Code Generation)**: In compiler design, assigning a limited number of physical registers to program variables such that no two variables that are "live" at the same time share a register is equivalent to **Graph Coloring**, which is NP-Complete. This is why real compilers use heuristics rather than exact optimal algorithms.
- **Vertex Cover**: Does a graph have a set of at most k vertices such that every edge has at least one endpoint in the set? (NP-Complete.)
- **Subset Sum**: Given a set of integers and a target sum, does some subset add up exactly to the target? (NP-Complete, and pseudo-polynomial solvable via dynamic programming.)

## Related Diagram

**Relationship between complexity classes:**

```
                 NP-Hard
             ___________________
            /                   \
           /        NP           \
          /      _________        \
         /      /  NP-    \        \
        /      /  Complete \        \
       |      |    (SAT,    |        |
       |   P  |   Vertex    |        |
       |      |   Cover,    |        |
        \      \ Ham-Cycle)/        /
         \      \_________/        /
          \___________________ ___/
```

**Reduction chain (Cook's Theorem as the root):**

```
SAT (Cook-Levin, proven NP-Complete directly)
   |  polynomial-time reduction
   v
3-SAT --> Vertex Cover --> Hamiltonian Cycle
   |
   v
Subset Sum, Partition, Graph Coloring, ...
```

Each arrow represents "problem X reduces to problem Y in polynomial time," meaning if Y could be solved efficiently, so could X.

## Computer Code

### 1. Cook's Theorem  Boolean Satisfiability (SAT) Solver

Cook's Theorem states that SAT is NP-Complete. Here we implement a brute-force SAT solver: given a Boolean formula (expressed as a function over variable assignments), it tries every possible truth assignment (2^n possibilities) to check satisfiability  directly illustrating why SAT verification is easy (polynomial) but solving is believed to be exponential in the worst case.

In [1]:
from itertools import product

def sat_solver(num_vars, clauses):
    """
    clauses: list of clauses, where each clause is a list of literals.
    A positive integer i means variable x_i, a negative integer -i means NOT x_i.
    Example: (x1 OR NOT x2) AND (x2 OR x3)  ->  [[1, -2], [2, 3]]
    Returns a satisfying assignment (dict) if one exists, else None.
    """
    for assignment in product([True, False], repeat=num_vars):
        # assignment[i-1] corresponds to variable x_i
        if all(
            any((assignment[abs(lit) - 1] if lit > 0 else not assignment[abs(lit) - 1])
                for lit in clause)
            for clause in clauses
        ):
            return {f"x{i+1}": val for i, val in enumerate(assignment)}
    return None

# Example formula: (x1 OR x2) AND (NOT x1 OR x3) AND (NOT x2 OR NOT x3)
num_vars = 3
clauses = [[1, 2], [-1, 3], [-2, -3]]

result = sat_solver(num_vars, clauses)
print(f"Formula (CNF clauses): {clauses}")
if result:
    print(f"Satisfiable! Example assignment: {result}")
else:
    print("Formula is UNSATISFIABLE")

Formula (CNF clauses): [[1, 2], [-1, 3], [-2, -3]]
Satisfiable! Example assignment: {'x1': True, 'x2': False, 'x3': True}


### 2. NP-Hard Graph Problem — Hamiltonian Cycle

Determines whether a graph contains a Hamiltonian Cycle (a cycle visiting every vertex exactly once) using backtracking. This problem is NP-Complete; verifying a given cycle is polynomial, but finding one in the worst case requires exploring an exponential number of vertex orderings.

In [2]:
def hamiltonian_cycle(graph, n):
    path = [-1] * n
    path[0] = 0

    def is_safe(v, pos):
        if graph[path[pos - 1]][v] == 0:
            return False
        if v in path:
            return False
        return True

    def solve(pos):
        if pos == n:
            return graph[path[pos - 1]][path[0]] == 1
        for v in range(1, n):
            if is_safe(v, pos):
                path[pos] = v
                if solve(pos + 1):
                    return True
                path[pos] = -1
        return False

    if solve(1):
        return path + [path[0]]
    return None

# Adjacency matrix representation of a graph with 5 vertices
graph = [
    [0, 1, 0, 1, 0],
    [1, 0, 1, 1, 1],
    [0, 1, 0, 0, 1],
    [1, 1, 0, 0, 1],
    [0, 1, 1, 1, 0],
]

result = hamiltonian_cycle(graph, len(graph))
if result:
    print(f"Hamiltonian Cycle found: {' -> '.join(map(str, result))}")
else:
    print("No Hamiltonian Cycle exists in this graph")

Hamiltonian Cycle found: 0 -> 1 -> 2 -> 4 -> 3 -> 0


### 3. NP-Hard Scheduling Problem — Minimum Makespan Scheduling

Given a set of jobs with processing times and a fixed number of identical machines, this brute-force solution tries every possible assignment of jobs to machines to find the assignment that minimizes the makespan (maximum load on any machine). This scheduling problem is NP-Hard in general (related to the Partition Problem).

In [3]:
from itertools import product

def min_makespan_scheduling(jobs, num_machines):
    """
    jobs: list of processing times
    num_machines: number of identical machines
    Returns the best assignment and the minimum makespan (brute force).
    """
    best_makespan = float('inf')
    best_assignment = None

    for assignment in product(range(num_machines), repeat=len(jobs)):
        loads = [0] * num_machines
        for job_time, machine in zip(jobs, assignment):
            loads[machine] += job_time
        makespan = max(loads)
        if makespan < best_makespan:
            best_makespan = makespan
            best_assignment = assignment

    return best_assignment, best_makespan

jobs = [4, 5, 6, 3, 7]
num_machines = 2

assignment, makespan = min_makespan_scheduling(jobs, num_machines)
print(f"Jobs (processing times): {jobs}")
print(f"Number of machines: {num_machines}")
print(f"Best job-to-machine assignment: {assignment}")
print(f"Minimum makespan: {makespan}")

for m in range(num_machines):
    assigned_jobs = [jobs[i] for i in range(len(jobs)) if assignment[i] == m]
    print(f"  Machine {m}: jobs {assigned_jobs}, load = {sum(assigned_jobs)}")

Jobs (processing times): [4, 5, 6, 3, 7]
Number of machines: 2
Best job-to-machine assignment: (0, 0, 1, 0, 1)
Minimum makespan: 13
  Machine 0: jobs [4, 5, 3], load = 12
  Machine 1: jobs [6, 7], load = 13


### 4. NP-Hard Code Generation Problem — Register Allocation via Graph Coloring

In compiler code generation, assigning a limited number of CPU registers to variables (such that two variables that are simultaneously "live" never share a register) is equivalent to coloring an **interference graph** with a limited number of colors. Graph Coloring is NP-Complete, so this backtracking solution illustrates why real compilers rely on heuristics (e.g., Chaitin's algorithm) rather than exact solutions for large programs.

In [4]:
def register_allocation(interference_graph, variables, num_registers):
    """
    interference_graph: dict {variable: set of variables it interferes with}
    Returns a mapping {variable: register_number} if allocation is possible, else None.
    """
    assignment = {}

    def is_safe(var, reg):
        for neighbor in interference_graph[var]:
            if neighbor in assignment and assignment[neighbor] == reg:
                return False
        return True

    def solve(index):
        if index == len(variables):
            return True
        var = variables[index]
        for reg in range(num_registers):
            if is_safe(var, reg):
                assignment[var] = reg
                if solve(index + 1):
                    return True
                del assignment[var]
        return False

    if solve(0):
        return assignment
    return None

# Interference graph: which variables are "live" at the same time
variables = ['a', 'b', 'c', 'd']
interference_graph = {
    'a': {'b', 'c'},
    'b': {'a', 'c', 'd'},
    'c': {'a', 'b'},
    'd': {'b'},
}
num_registers = 3

result = register_allocation(interference_graph, variables, num_registers)
print(f"Variables: {variables}")
print(f"Interference graph: {interference_graph}")
print(f"Number of available registers: {num_registers}")
if result:
    print(f"Register allocation found: {result}")
else:
    print(f"No valid allocation with only {num_registers} registers (spilling required)")

Variables: ['a', 'b', 'c', 'd']
Interference graph: {'a': {'b', 'c'}, 'b': {'a', 'd', 'c'}, 'c': {'b', 'a'}, 'd': {'b'}}
Number of available registers: 3
Register allocation found: {'a': 0, 'b': 1, 'c': 2, 'd': 0}


### 5. Vertex Cover Problem

In [5]:
from itertools import combinations

def vertex_cover(edges, num_vertices, k):
    """
    Checks if a vertex cover of size at most k exists (brute force over subsets).
    Returns the cover if found, else None.
    """
    vertices = list(range(num_vertices))
    for size in range(1, k + 1):
        for subset in combinations(vertices, size):
            subset_set = set(subset)
            if all(u in subset_set or v in subset_set for u, v in edges):
                return subset
    return None

edges = [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
num_vertices = 5
k = 2

result = vertex_cover(edges, num_vertices, k)
print(f"Edges: {edges}")
print(f"Checking for a vertex cover of size <= {k}")
if result:
    print(f"Vertex cover found: {result}")
else:
    print(f"No vertex cover of size <= {k} exists")

Edges: [(0, 1), (0, 2), (1, 2), (1, 3), (2, 4), (3, 4)]
Checking for a vertex cover of size <= 2
No vertex cover of size <= 2 exists


### 6. Subset Sum Problem

In [6]:
def subset_sum(arr, target):
    """
    Brute-force backtracking search for a subset summing exactly to target.
    Returns the subset if found, else None.
    """
    n = len(arr)

    def backtrack(index, current_sum, current_subset):
        if current_sum == target:
            return list(current_subset)
        if index == n or current_sum > target:
            return None
        # include arr[index]
        current_subset.append(arr[index])
        result = backtrack(index + 1, current_sum + arr[index], current_subset)
        if result is not None:
            return result
        current_subset.pop()
        # exclude arr[index]
        return backtrack(index + 1, current_sum, current_subset)

    return backtrack(0, 0, [])

arr = [3, 34, 4, 12, 5, 2]
target = 9

result = subset_sum(arr, target)
print(f"Set: {arr}")
print(f"Target sum: {target}")
if result:
    print(f"Subset found: {result}, sum = {sum(result)}")
else:
    print("No subset sums to the target")

Set: [3, 34, 4, 12, 5, 2]
Target sum: 9
Subset found: [3, 4, 2], sum = 9


## Analysis of the Algorithms

- **SAT Solver (Cook's Theorem)**: Brute force runs in O(2^n · m) time for n variables and m clauses, since every possible truth assignment must be checked in the worst case. This exponential blow-up is exactly why SAT is hard to *solve*, even though a proposed solution can be *verified* in O(m) time — the essence of the P vs NP question.
- **Hamiltonian Cycle**: The backtracking solution runs in O(n!) in the worst case, since it explores permutations of vertices while pruning invalid partial paths early.
- **Minimum Makespan Scheduling**: The brute-force assignment search runs in O(k^n) for n jobs and k machines, since every job can be assigned to any machine; this becomes intractable quickly as job count grows, reflecting the underlying NP-Hardness (related to Partition, which is NP-Complete).
- **Register Allocation (Graph Coloring)**: Backtracking coloring runs in O(k^n) in the worst case for n variables and k registers, illustrating why real compilers use greedy/heuristic coloring instead of exact algorithms for large programs.
- **Vertex Cover**: The brute-force search over subsets runs in O(n^k · E) for checking covers up to size k, which is exponential in k; a well-known result shows Vertex Cover is fixed-parameter tractable in k, but remains NP-Complete in general.
- **Subset Sum**: The backtracking approach runs in O(2^n) in the worst case; note that Subset Sum also has a pseudo-polynomial O(n · target) dynamic programming solution, which is efficient when the target sum is small but not polynomial in the input's bit-length.

## Discussion and Conclusion

This lab explored the theoretical foundations of computational complexity through the lens of Cook's Theorem and several classical NP-Hard / NP-Complete problems. Implementing a brute-force SAT solver made concrete the idea at the heart of the Cook-Levin Theorem: satisfiability is easy to *verify* but believed to be hard to *solve* in the worst case, and every problem in NP can, in principle, be reduced to SAT.

The Hamiltonian Cycle, Vertex Cover, and Subset Sum implementations each showed exponential-time behavior in their brute-force/backtracking forms, consistent with their status as NP-Complete problems — no polynomial-time algorithm is known (or believed to exist) for solving them exactly in the general case.

The scheduling and register allocation examples illustrated that NP-Hardness is not just a theoretical concern but appears directly in practical systems: operating systems must schedule jobs across machines, and compilers must allocate a limited number of registers to program variables. In both cases, real-world systems rely on heuristics or approximation algorithms rather than exact exponential-time solutions, since guaranteed optimality is computationally infeasible for large inputs.

Overall, this lab reinforced the practical significance of the P vs NP question: while these problems can be solved exactly for small inputs (as demonstrated here), any approach that guarantees optimal solutions for all such problems in polynomial time would resolve one of the most important open problems in computer science.